In [74]:
import pandas as pd

df = pd.read_csv("thermopoints.csv", delimiter=";")
df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660254 entries, 0 to 660253
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   dt         660252 non-null  object 
 1   type_name  660254 non-null  object 
 2   type_id    660254 non-null  int64  
 3   lon        660254 non-null  float64
 4   lat        660254 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 25.2+ MB


,dt,type_name,type_id,lon,lat
0,NaN,Природный пожар,4,131.5866,47.8662
1,NaN,Природный пожар,4,131.5885,47.8809
2,2012-03-13,Лесной пожар,3,131.9871,48.4973
3,2012-03-13,Природный пожар,4,131.9031,43.6277
4,2012-03-13,Природный пожар,4,131.5706,47.8581
...,...,...,...,...,...
660249,2021-09-10,Лесной пожар,3,118.5451,64.7475
660250,2021-09-10,Лесной пожар,3,118.3046,64.7629
660251,2021-09-10,Лесной пожар,3,117.9681,65.7394
660252,2021-09-10,Лесной пожар,3,119.0462,64.7541


In [75]:
df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660254 entries, 0 to 660253
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   dt         660252 non-null  datetime64[ns]
 1   type_name  660254 non-null  object        
 2   type_id    660254 non-null  int64         
 3   lon        660254 non-null  float64       
 4   lat        660254 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 25.2+ MB


In [76]:
df.isna().sum()

dt           2
type_name    0
type_id      0
lon          0
lat          0
dtype: int64

In [77]:
df['dt'] = df['dt'].ffill().bfill()
df

,dt,type_name,type_id,lon,lat
0,2012-03-13,Природный пожар,4,131.5866,47.8662
1,2012-03-13,Природный пожар,4,131.5885,47.8809
2,2012-03-13,Лесной пожар,3,131.9871,48.4973
3,2012-03-13,Природный пожар,4,131.9031,43.6277
4,2012-03-13,Природный пожар,4,131.5706,47.8581
...,...,...,...,...,...
660249,2021-09-10,Лесной пожар,3,118.5451,64.7475
660250,2021-09-10,Лесной пожар,3,118.3046,64.7629
660251,2021-09-10,Лесной пожар,3,117.9681,65.7394
660252,2021-09-10,Лесной пожар,3,119.0462,64.7541


In [78]:
df.isna().sum()

dt           0
type_name    0
type_id      0
lon          0
lat          0
dtype: int64

In [79]:
df.duplicated().sum()

np.int64(1466)

In [80]:
df.drop_duplicates(inplace=True)

In [81]:
df.duplicated().sum()

np.int64(0)

In [82]:
df.describe()

,dt,type_id,lon,lat
count,658788,658788.000000,658788.000000,658788.000000
mean,2018-05-20 06:18:55.496092928,2.986244,107.753779,57.036427
min,2012-03-13 00:00:00,1.000000,19.862200,41.262800
25%,2016-04-18 00:00:00,3.000000,90.748850,52.536800
50%,2019-04-27 00:00:00,3.000000,114.911700,56.458100
75%,2020-08-21 00:00:00,4.000000,130.629300,62.467500
max,2021-09-10 00:00:00,5.000000,178.435100,72.741400
std,NaN,1.305009,29.411397,6.302507


In [83]:
import rasterio
import os
import glob

# Укажите путь к папке
images_folder = 'foto'

metadata_list = []

for image_path in glob.glob(os.path.join(images_folder, '*.*')):
    try:
        with rasterio.open(image_path) as src:
            meta = src.meta.copy()
            # Добавляем имя файла
            meta['file_name'] = os.path.basename(image_path)
            # Можно извлечь геопространственные тегы
            meta['crs'] = src.crs
            meta['bounds'] = src.bounds
            # Размер
            meta['width'] = src.width
            meta['height'] = src.height
            # Дата (если есть в метаданных файла)
            # rasterio не всегда извлекает дату, зависит от формата
            # Можно получить системную дату создания файла
            try:
                import datetime
                import time
                creation_time = os.path.getctime(image_path)
                meta['creation_date'] = datetime.datetime.fromtimestamp(creation_time).strftime('%Y-%m-%d %H:%M:%S')
            except:
                meta['creation_date'] = 'Неизвестна'
            metadata_list.append(meta)
    except Exception as e:
        print(f"Ошибка при чтении {image_path}: {e}")

# Выводим
import pandas as pd
df_metadata = pd.DataFrame(metadata_list)
print(df)

               dt        type_name  type_id       lon      lat
0      2012-03-13  Природный пожар        4  131.5866  47.8662
1      2012-03-13  Природный пожар        4  131.5885  47.8809
2      2012-03-13     Лесной пожар        3  131.9871  48.4973
3      2012-03-13  Природный пожар        4  131.9031  43.6277
4      2012-03-13  Природный пожар        4  131.5706  47.8581
...           ...              ...      ...       ...      ...
660249 2021-09-10     Лесной пожар        3  118.5451  64.7475
660250 2021-09-10     Лесной пожар        3  118.3046  64.7629
660251 2021-09-10     Лесной пожар        3  117.9681  65.7394
660252 2021-09-10     Лесной пожар        3  119.0462  64.7541
660253 2021-09-10     Лесной пожар        3  114.6958  64.5745

[658788 rows x 5 columns]


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [84]:
df_metadata

,driver,dtype,nodata,width,height,count,crs,transform,file_name,bounds,creation_date
0,WEBP,uint8,None,614,922,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3101.jpg,"(0.0, 922.0, 614.0, 0.0)",2026-04-14 20:08:52
1,WEBP,uint8,None,1280,852,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3103.jpg,"(0.0, 852.0, 1280.0, 0.0)",2026-04-14 20:08:52
2,WEBP,uint8,None,1280,852,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3106.jpg,"(0.0, 852.0, 1280.0, 0.0)",2026-04-14 20:08:52
3,WEBP,uint8,None,634,922,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3107.jpg,"(0.0, 922.0, 634.0, 0.0)",2026-04-14 20:08:52
4,GTiff,uint8,None,7671,7791,3,"(proj, zone, datum, units, no_defs)","(30.0, 0.0, 570285.0, 0.0, -30.0, 5216415.0, 0...",LC08_L1TP_173028_20171231_20200902_02_T1_refl.tif,"(570285.0, 4982685.0, 800415.0, 5216415.0)",2026-04-13 23:32:37


In [85]:
from pyproj import Transformer
import re

# 1. Настройка трансформера (метры -> градусы)
tf = Transformer.from_crs("EPSG:32637", "EPSG:4326", always_xy=True)

def get_lon_lat(b):
    # Извлекаем числа из строки или объекта
    nums = [float(x) for x in re.findall(r"[-+]?\d*\.\d+|\d+", str(b))]
    # Считаем центр (left+right)/2 и (bottom+top)/2
    x, y = (nums[0] + nums[2]) / 2, (nums[1] + nums[3]) / 2
    return tf.transform(x, y)

# 2. Одной строкой создаем обе колонки
df_metadata[['longitude', 'latitude']] = df_metadata['bounds'].apply(lambda x: get_lon_lat(x)).tolist()

print(df_metadata[['file_name', 'longitude', 'latitude']].head())


                                           file_name  longitude   latitude
0                                       IMG_3101.jpg  34.514007   0.004158
1                                       IMG_3103.jpg  34.516990   0.003842
2                                       IMG_3106.jpg  34.516990   0.003842
3                                       IMG_3107.jpg  34.514096   0.004158
4  LC08_L1TP_173028_20171231_20200902_02_T1_refl.tif  41.394782  46.024429


In [86]:
df_metadata

,driver,dtype,nodata,width,height,count,crs,transform,file_name,bounds,creation_date,longitude,latitude
0,WEBP,uint8,None,614,922,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3101.jpg,"(0.0, 922.0, 614.0, 0.0)",2026-04-14 20:08:52,34.514007,0.004158
1,WEBP,uint8,None,1280,852,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3103.jpg,"(0.0, 852.0, 1280.0, 0.0)",2026-04-14 20:08:52,34.516990,0.003842
2,WEBP,uint8,None,1280,852,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3106.jpg,"(0.0, 852.0, 1280.0, 0.0)",2026-04-14 20:08:52,34.516990,0.003842
3,WEBP,uint8,None,634,922,3,None,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",IMG_3107.jpg,"(0.0, 922.0, 634.0, 0.0)",2026-04-14 20:08:52,34.514096,0.004158
4,GTiff,uint8,None,7671,7791,3,"(proj, zone, datum, units, no_defs)","(30.0, 0.0, 570285.0, 0.0, -30.0, 5216415.0, 0...",LC08_L1TP_173028_20171231_20200902_02_T1_refl.tif,"(570285.0, 4982685.0, 800415.0, 5216415.0)",2026-04-13 23:32:37,41.394782,46.024429
